In [ ]:
import pandas as pd
import os
from dotenv import dotenv_values
from parsers import load_positions

DATA_DIR = os.path.dirname(os.path.abspath('mystocks.ipynb'))
DATA_DIR_DATA = DATA_DIR + "/accounts"

_env = dotenv_values(os.path.join(DATA_DIR, '.env'))
acc_names = {k.replace('ACC_', ''): v for k, v in _env.items() if k.startswith('ACC_')}

exclude = {"earnings.csv", "historical.csv", "portfolio.csv",
           "sectors.csv", "file_clean.csv", "History_for_Account_226998197.csv",
           "History_for_Account_236369828.csv"}

accounts = load_positions(DATA_DIR_DATA, exclude=exclude)

for acct in sorted(accounts):
    print(f"\n{'='*55}")
    print(f" Account: {acct}  {acc_names.get(acct, 'found none')}")
    print(f"{'='*55}")
    print(accounts[acct].to_string(float_format='%.4f'))


In [ ]:
# Merge all accounts into one DataFrame, grouped by Symbol
from analytics import merge_accounts

combined = merge_accounts(accounts)
combined


In [ ]:
# ── Trade History & Cost Basis ────────────────────────────────────────────────
from parsers.transactions import load_transactions, load_realized_lots
from analytics import load_fallback_cost_basis, compute_cost_basis, closed_positions_summary, CUTOFF, DEFAULT_BUY

BUYSELL_DIR = os.path.join(DATA_DIR, 'buysell')

tx_df = load_transactions(BUYSELL_DIR)
sold_df = load_realized_lots(BUYSELL_DIR, CUTOFF)
fid_cost = load_fallback_cost_basis(DATA_DIR_DATA)

combined = compute_cost_basis(combined, tx_df, fid_cost, CUTOFF, DEFAULT_BUY)

# ── Print: current holdings with entry dates ───────────────────────────────
hold_out = combined[combined.index != 'cash'][
    ['First_Buy_Date', 'Days_Held', 'Avg_Buy_Price', 'Current_Price', 'Market_Value', 'Cost_Basis_Source']
].copy()
hold_out['Return_%'] = (
    (combined['Current_Price'] / hold_out['Avg_Buy_Price'] - 1) * 100
).round(2)
hold_out = hold_out.sort_values('First_Buy_Date')

print(f'\n{"═"*72}')
print('  Current Holdings — Entry Dates & Return vs Avg Cost')
print(f'{"═"*72}')
print(hold_out.to_string(float_format='%.2f'))

# ── Print: closed positions summary ────────────────────────────────────────
closed_summ = closed_positions_summary(sold_df)
if closed_summ:
    summ_df = pd.DataFrame(closed_summ).set_index('Symbol')
    summ_df['First_Buy'] = pd.to_datetime(summ_df['First_Buy']).dt.date
    summ_df['Last_Sell'] = pd.to_datetime(summ_df['Last_Sell']).dt.date
    print(f'\n{"═"*72}')
    print('  Closed Positions — Realized Gain/Loss (2023+)')
    print(f'{"═"*72}')
    print(summ_df.to_string())


In [ ]:
# ── Capital Tracking & Cash-Adjusted Performance ──────────────────────────────
from analytics import capital_performance

# NOTE: REINVEST transactions no longer count as "Bought" capital deployed —
# dividend reinvestment isn't new capital in, even though it adds to cost
# basis (see Cost_Basis_Source / Avg_Buy_Price above). Tracked separately below.
cap = capital_performance(combined, tx_df, sold_df)

annual = pd.DataFrame(cap['annual_activity']).set_index('Year')
print(f'\n{"═"*70}')
print('  Annual Capital Activity  (proxy: net securities purchased; excludes REINVEST)')
print(f'{"═"*70}')
print(annual.to_string(float_format='${:,.0f}'.format))

if cap['dividends_reinvested']:
    divs = pd.DataFrame(cap['dividends_reinvested']).set_index('Year')
    print('\n  Dividends reinvested by year (not counted as new capital deployed):')
    print(divs.to_string())

# ── Current holdings: cost basis vs market value ───────────────────────────
holdings_cb = pd.DataFrame(cap['holdings_cost_basis']).set_index('Symbol')
print(f'\n{"═"*70}')
print('  Current Holdings — Cost Basis vs Market Value')
print(f'{"═"*70}')
print(holdings_cb.to_string(float_format='%.2f'))

t = cap['totals']
print(f'\n  {"TOTAL":<20} Cost: ${t["cost_basis"]:>12,.2f}   MktVal: ${t["market_value"]:>12,.2f}   Unreal G/L: ${t["unrealized_gl"]:>12,.2f}')

# ── Overall return on invested capital ─────────────────────────────────────
print(f'\n{"═"*70}')
print('  Overall Return on Invested Capital')
print(f'{"═"*70}')
print(f'  Current holdings cost basis : ${t["cost_basis"]:>12,.2f}')
print(f'  Closed positions cost basis : ${t["closed_cost_basis"]:>12,.2f}')
print(f'  ─────────────────────────────────────────')
print(f'  Total capital invested      : ${t["total_invested"]:>12,.2f}')
print(f'')
print(f'  Unrealized gain/loss        : ${t["unrealized_gl"]:>12,.2f}')
print(f'  Realized gain/loss (2023+)  : ${t["realized_gl"]:>12,.2f}')
print(f'  ─────────────────────────────────────────')
print(f'  Total P&L                   : ${t["total_pl"]:>12,.2f}')
print(f'  Return on invested capital  :   {t["roic_pct"]:>10.2f}%')
print(f'  (normalized for capital changes — not distorted by contributions)')


In [ ]:
from enrich import refresh_historical, symbol_metrics, risk_free_rate

HIST_FILE = os.path.join(DATA_DIR, 'historical.csv')
hist_df = refresh_historical(combined.index, HIST_FILE)
print(f'historical.csv: {len(hist_df)} rows, {len(hist_df.columns)} symbols')

rf_annual = risk_free_rate()
print(f'Risk-free rate: {rf_annual:.2%}')

metrics_df = symbol_metrics(combined, hist_df, rf_annual)

# ── Push live prices + Market_Value back into combined ────────────────────────
live_prices = metrics_df['Current_Price'].dropna()
combined.loc[live_prices.index, 'Current_Price'] = live_prices
combined.loc[combined.index != 'cash', 'Market_Value'] = (
    combined.loc[combined.index != 'cash', 'Quantity']
    * combined.loc[combined.index != 'cash', 'Current_Price']
)

# ── Join all other metrics ────────────────────────────────────────────────────
for col in metrics_df.columns:
    if col != 'Current_Price':
        combined[col] = metrics_df[col]

combined


In [ ]:
# ── Snapshot Comparison — Infer Missing Trades ────────────────────────────────
# Headless-safe: unresolved trades are always logged to unknown_trades.csv,
# never prompted for interactively (this is what lets run_pipeline.py run
# unattended via cron/systemd).
from analytics import collect_snapshots, infer_missing_trades, compute_cost_basis

PAST_DIR     = os.path.join(DATA_DIR, 'past')
UNKNOWN_FILE = os.path.join(DATA_DIR, 'unknown_trades.csv')

snap_map = collect_snapshots(PAST_DIR, DATA_DIR_DATA)
inferred = infer_missing_trades(snap_map, tx_df, hist_df, UNKNOWN_FILE)

if not inferred.empty:
    tx_df = pd.concat([tx_df, inferred], ignore_index=True).sort_values('Date').reset_index(drop=True)
    print(f'\n{"═"*72}')
    print(f'  Inferred {len(inferred)} trade(s) from snapshot comparison')
    print(f'{"═"*72}')
    print(inferred[['Symbol','Date','Action','Quantity','Price','Account']].to_string(index=False))

    combined = compute_cost_basis(combined, tx_df, fid_cost, CUTOFF, DEFAULT_BUY)
else:
    print('  Snapshot comparison: no unexplained position changes found.')


In [7]:
PORT_FILE  = os.path.join(DATA_DIR, 'portfolio.csv')
combined.to_csv(PORT_FILE)

In [8]:
eq = combined[combined.index != 'cash'].copy()
N  = 10

def show(title, df, cols):
    print(f'\n{chr(8212)*60}')
    print(f'  {title}')
    print(f'{chr(8212)*60}')
    print(df[cols].to_string())

# Highest Trailing P/E
top_tpe = eq['Trailing_PE'].dropna().nlargest(N)
show('WARNING  Highest Trailing P/E (most expensive on earnings)',
     eq.loc[top_tpe.index], ['Trailing_PE', 'Forward_PE', 'Current_Price', 'Market_Value'])

# Highest Forward P/E
top_fpe = eq['Forward_PE'].dropna().nlargest(N)
show('WARNING  Highest Forward P/E (market expects slow earnings growth)',
     eq.loc[top_fpe.index], ['Forward_PE', 'Trailing_PE', 'Current_Price', 'Market_Value'])

# Worst 3m Performance
worst_3m = eq['Gain_3m'].dropna().nsmallest(N)
show('WARNING  Worst 3-Month Performance',
     eq.loc[worst_3m.index], ['Gain_3m', 'Gain_6m', 'Gain_1yr', 'Sharpe_3m', 'Current_Price'])

# Lowest Forward P/E
low_fpe = eq['Forward_PE'].dropna().nsmallest(N)
show('OK  Lowest Forward P/E (potentially undervalued)',
     eq.loc[low_fpe.index], ['Forward_PE', 'Trailing_PE', 'Current_Price', 'Target_Mean', 'Market_Value'])

# Best 3m Performance
best_3m = eq['Gain_3m'].dropna().nlargest(N)
show('OK  Best 3-Month Performance',
     eq.loc[best_3m.index], ['Gain_3m', 'Gain_6m', 'Gain_1yr', 'Sharpe_3m', 'Current_Price'])



————————————————————————————————————————————————————————————
  WARNING  Highest Trailing P/E (most expensive on earnings)
————————————————————————————————————————————————————————————
        Trailing_PE  Forward_PE  Current_Price  Market_Value
Symbol                                                      
COHR         151.55       37.84         312.19       1248.76
SHOP         120.84       50.79         118.42        236.84
MPWR          99.04       45.71        1398.45       1398.45
STX           84.71       31.89         908.10        908.10
SBUX          79.98       34.57         103.98      10398.00
GLW           78.20       35.91         154.06        462.18
VRT           76.24       33.95         301.16       3312.76
TER           69.45       35.93         369.46       3694.60
AVGO          64.52       20.39         396.81      15872.40
XYZ           62.98       15.30          77.46        929.52

————————————————————————————————————————————————————————————
  WARNING  Highest For

In [ ]:
# ── Sector / Cap-size / Vol-tier classification ───────────────────────────────
from enrich import classify_sectors

SECTOR_CSV = os.path.join(DATA_DIR, 'sectors.csv')
sector_data = classify_sectors(combined, SECTOR_CSV)

for col in ('Quote_Type', 'Sector', 'MarketCap', 'Cap_Tier', 'Vol_Tier'):
    combined.loc[combined.index != 'cash', col] = sector_data[col]

eq = combined[combined.index != 'cash'].copy()

GAIN_COLS = ['Gain_3m', 'Gain_6m', 'Gain_1yr']

def sector_summary(df, group_col):
    g = df.groupby(group_col)
    mv  = g['Market_Value'].sum().rename('Total_Market_Value')
    wgains = {}
    for gc in GAIN_COLS:
        sub = df[['Market_Value', gc, group_col]].dropna(subset=[gc])
        wgains[gc] = (
            sub.groupby(group_col)
               .apply(lambda x: (x[gc] * x['Market_Value']).sum() / x['Market_Value'].sum(),
                      include_groups=False)
        )
    gain_df = pd.DataFrame(wgains).round(2)
    result  = pd.concat([mv, gain_df], axis=1).sort_values('Total_Market_Value', ascending=False)
    result['Total_Market_Value'] = result['Total_Market_Value'].map('${:,.0f}'.format)
    return result

def print_summary(title, df):
    print(f'\n{"═"*65}')
    print(f'  {title}')
    print(f'{"═"*65}')
    print(df.to_string())

print_summary('By GICS Sector',       sector_summary(eq, 'Sector'))
print_summary('By Market-Cap Tier',   sector_summary(eq, 'Cap_Tier'))
print_summary('By Volatility Tier',   sector_summary(eq, 'Vol_Tier'))


In [ ]:
# ── Analyst Target Analysis ───────────────────────────────────────────────────
from enrich import analyst_targets

tgt = analyst_targets(combined)

def show(title, df, cols):
    print(f'\n{chr(8213)*62}')
    print(f'  {title}')
    print(f'{chr(8213)*62}')
    print(df[cols].to_string())

# 1. Analyst median target BELOW current price
overvalued = tgt[tgt['Target_Median'] < tgt['Current_Price']].sort_values('Target_Upside')
show(
    'WARNING  Analyst Median Target BELOW Current Price',
    overvalued,
    ['Current_Price', 'Target_Median', 'Target_Upside', 'Target_Low', 'Target_High', 'Num_Analysts']
)

# 2. Biggest upside to analyst median target (top 10)
most_upside = tgt.nlargest(10, 'Target_Upside')
show(
    'OK  Most Upside to Analyst Median Target (top 10)',
    most_upside,
    ['Current_Price', 'Target_Median', 'Target_Upside', 'Target_High', 'Num_Analysts']
)

# 3. Tightest analyst consensus (top 10 narrowest spread)
tightest = tgt.nsmallest(10, 'Target_Spread')
show(
    'OK  Tightest Analyst Consensus  (Target_High - Target_Low) / Target_Median  (top 10)',
    tightest,
    ['Current_Price', 'Target_Median', 'Target_Upside', 'Target_Spread', 'Num_Analysts']
)


In [ ]:
# ── Earnings dates, analyst recommendations, upgrades/downgrades ──────────────
from datetime import date
from enrich import earnings_and_recommendations
import warnings
warnings.filterwarnings('ignore')

EARN_FILE = os.path.join(DATA_DIR, 'earnings.csv')
today   = pd.Timestamp(date.today())
symbols = combined.index[combined.index != 'cash'].tolist()

earn_cache, recs_df, upgrades = earnings_and_recommendations(combined, EARN_FILE)
print(f'Saved earnings cache: {EARN_FILE}  ({len(earn_cache)} symbols)')

for col in recs_df.columns:
    combined[col] = recs_df[col]


In [ ]:
# ── Export all tables to JSON for the viewer app ─────────────────────────────
from analytics import export_app_data

APP_DATA = os.path.join(DATA_DIR, 'app_data')
UD_FILE  = os.path.join(DATA_DIR, 'upgrades.csv')

export_app_data(APP_DATA, accounts, combined, earn_file=EARN_FILE, upgrades_file=UD_FILE)

print(f"Saved app data to {APP_DATA}/")
for fn in sorted(os.listdir(APP_DATA)):
    path = os.path.join(APP_DATA, fn)
    print(f"  {fn}  ({os.path.getsize(path):,} bytes)")


In [13]:
# ── Print: next earnings by date ──────────────────────────────────────────────
upcoming = (
    earn_cache[earn_cache['Next_Earnings'].notna() & (earn_cache['Next_Earnings'] > today)]
    .sort_values('Next_Earnings')
)
# Only show symbols we own
upcoming = upcoming[upcoming.index.isin(symbols)]

print(f'\n{"═"*62}')
print('  Upcoming Earnings (soonest first)')
print(f'{"═"*62}')
print(upcoming[['Next_Earnings','EPS_Est','Rev_Est_High','Rev_Est_Low']].to_string())

# ── Print: buy/sell/hold summary ──────────────────────────────────────────────
if not recs_df.empty:
    print(f'\n{"═"*62}')
    print('  Analyst Recommendations (current month)')
    print(f'{"═"*62}')
    print(
        recs_df.sort_values(['Strong_Buy','Buy'], ascending=False)
               .to_string()
    )

# ── Print: recent upgrades/downgrades ────────────────────────────────────────
if upgrades:
    ud_all = pd.concat(upgrades).sort_values('GradeDate', ascending=False)
    print(f'\n{"═"*62}')
    print('  Upgrades / Downgrades  (last 90 days)')
    print(f'{"═"*62}')
    print(ud_all.to_string(index=False))



══════════════════════════════════════════════════════════════
  Upcoming Earnings (soonest first)
══════════════════════════════════════════════════════════════
       Next_Earnings   EPS_Est  Rev_Est_High   Rev_Est_Low
Symbol                                                    
SAP       2026-07-23   2.01953  9.963000e+09  9.691400e+09
INTC      2026-07-23   0.20977  1.486000e+10  1.420700e+10
GOOGL     2026-07-23   2.88458  1.220218e+11  1.136230e+11
TMUS      2026-07-23   2.67500  2.316100e+10  2.279700e+10
NDAQ      2026-07-23   0.94526  1.463000e+09  1.398000e+09
GOOG      2026-07-23   2.87790  1.201260e+11  1.136230e+11
AXP       2026-07-24   4.37935  1.970700e+10  1.945700e+10
NXPI      2026-07-27   3.52732  3.498807e+09  3.449893e+09
SEVN      2026-07-27   0.26000  9.105000e+06  9.100000e+06
AMKR      2026-07-27   0.47177  1.822606e+09  1.773000e+09
GLW       2026-07-28   0.75326  4.661174e+09  4.597326e+09
TXN       2026-07-28   1.94641  5.404000e+09  5.000000e+09
SPOT      2

In [14]:
earn_cache

,Next_Earnings,EPS_Est,Rev_Est_High,Rev_Est_Low
Symbol,,,,
AAPL,2026-07-30,1.89077,1.120000e+11,1.075010e+11
AMKR,2026-07-27,0.47177,1.822606e+09,1.773000e+09
AMZN,2026-07-30,1.81692,1.995570e+11,1.860000e+11
AVGO,2026-09-03,3.17944,3.753400e+10,2.497600e+10
AXP,2026-07-24,4.37935,1.970700e+10,1.945700e+10
...,...,...,...,...
SPMO,NaT,NaN,NaN,NaN
STX,2026-07-28,5.07530,3.559490e+09,3.447000e+09
THLLY,NaT,NaN,NaN,NaN


In [ ]:
# ── Modern Portfolio Theory — Core Metrics ────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from analytics import mpt_metrics

MPT_LOOKBACK = 252  # ~1 trading year
metrics = mpt_metrics(combined, hist_df, rf_annual, lookback=MPT_LOOKBACK)

mpt_syms = metrics['symbols']
n        = len(mpt_syms)
w0       = metrics['w0']
p_ret, p_vol, p_sr = metrics['p_ret'], metrics['p_vol'], metrics['p_sr']

print(f'{n} symbols in MPT universe')
print(f'\n{"═"*62}')
print(f'  Current Portfolio  ({n} equity symbols, 1-yr lookback)')
print(f'{"═"*62}')
print(f'  Expected Return (ann):  {p_ret:>9.2%}')
print(f'  Volatility (ann):       {p_vol:>9.2%}')
print(f'  Sharpe Ratio:           {p_sr:>9.3f}')
print(f'  Risk-free Rate:         {rf_annual:>9.2%}')
print(f'  Total Equity MV:        ${combined.loc[mpt_syms, "Market_Value"].sum():>12,.0f}')

# ── Beta & Jensen's Alpha vs SPY ──────────────────────────────────────────────
if 'beta_alpha' in metrics:
    ba_df = metrics['beta_alpha']
    combined.loc[ba_df.index, 'Beta']      = ba_df['Beta']
    combined.loc[ba_df.index, 'Alpha_pct'] = ba_df['Alpha_pct']

    print(f'  Portfolio Beta (vs SPY): {metrics["port_beta"]:>8.3f}')

    print(f'\n  High-beta positions (Beta > 1.5):')
    hb = ba_df[ba_df['Beta'] > 1.5].sort_values('Beta', ascending=False)
    print(hb.to_string() if not hb.empty else '  None')

    print(f"\n  Top 10 by Jensen's Alpha (ann %):")
    print(ba_df.nlargest(10, 'Alpha_pct').to_string())

# ── Risk Contribution ─────────────────────────────────────────────────────────
risk_df = metrics['risk_contrib']

print(f'\n{"═"*62}')
print(f'  Concentration — HHI: {metrics["hhi"]:.0f}/10000  |  Effective-N: {metrics["effective_n"]:.1f}')
print(f'{"═"*62}')
print(f'\n  Top 15 Risk Contributors (% of portfolio volatility):')
print(risk_df.nlargest(15, 'RiskContrib_pct').to_string())


In [ ]:
# ── Efficient Frontier (Monte Carlo + Optimization) ───────────────────────────
from analytics import efficient_frontier
from plotting import build_efficient_frontier_figure

ef = efficient_frontier(metrics, n_mc=3000, seed=42)

r_ms, v_ms, s_ms = ef['max_sharpe']['ret'], ef['max_sharpe']['vol'], ef['max_sharpe']['sharpe']
r_mv, v_mv, s_mv = ef['min_var']['ret'], ef['min_var']['vol'], ef['min_var']['sharpe']

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = build_efficient_frontier_figure(metrics, ef)
EF_PATH = os.path.join(DATA_DIR, 'efficient_frontier.png')
fig.savefig(EF_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved {EF_PATH}')

# ── Summary table ─────────────────────────────────────────────────────────────
print(f'\n{"═"*62}')
print(f'  Optimized vs Current')
print(f'{"═"*62}')
print(f'  {"Portfolio":<16} {"Return":>8} {"Vol":>8} {"Sharpe":>8}')
print(f'  {"-"*44}')
print(f'  {"Current":<16} {p_ret:>8.2%} {p_vol:>8.2%} {p_sr:>8.3f}')
print(f'  {"Max Sharpe":<16} {r_ms:>8.2%} {v_ms:>8.2%} {s_ms:>8.3f}')
print(f'  {"Min Variance":<16} {r_mv:>8.2%} {v_mv:>8.2%} {s_mv:>8.3f}')

print(f'\n  Max-Sharpe weights > 1%:')
ms_w = (ef['w_max_sharpe'] * 100).round(2)
print(ms_w[ms_w > 1].sort_values(ascending=False).to_string())


In [ ]:
# ── Correlation Heatmap (top 25 positions by market value) ────────────────────
from analytics import correlation_matrix, high_correlation_pairs
from plotting import build_correlation_heatmap_figure

rets = metrics['rets']
TOP_N = 25
corr, top_sym = correlation_matrix(combined, rets, top_n=TOP_N)

fig = build_correlation_heatmap_figure(corr, len(top_sym))
CORR_PATH = os.path.join(DATA_DIR, 'correlation_heatmap.png')
fig.savefig(CORR_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved {CORR_PATH}')

# ── Highly correlated pairs ────────────────────────────────────────────────────
THRESH = 0.85
high_corr = high_correlation_pairs(corr, top_sym, threshold=THRESH)
print(f'\n  Pairs with |correlation| > {THRESH}:')
if high_corr:
    for s1, s2, c in high_corr:
        print(f'  {s1:8s} ↔ {s2:8s}  {c:+.2f}')
else:
    print('  None found above threshold')
